# 10 · Minimum-cost flow, node potentials, and reduced costs

**Goal:** the general network LP. Shortest path, max flow, transportation and assignment are all special cases.

$$\min\ c^\top x\quad\text{s.t.}\quad Nx=b,\quad 0\le x\le u ,$$

with supplies $b_i>0$, demands $b_i<0$ and $\sum_ib_i=0$.

**Optimality via duals.** With node potentials $\pi$, the **reduced cost** of arc $(i,j)$ is $\bar c_{ij}=c_{ij}-\pi_i+\pi_j$. A feasible $x$ is optimal iff there is a $\pi$ with
* $\bar c_{ij}>0\Rightarrow x_{ij}=0$,
* $\bar c_{ij}<0\Rightarrow x_{ij}=u_{ij}$,
* $0<x_{ij}<u_{ij}\Rightarrow\bar c_{ij}=0$.

This is the certificate that network simplex works with.

In [ ]:
import numpy as np
import networkx as nx
from scipy.optimize import linprog

# (i, j, cost, capacity)
ARCS = [(0, 1, 2, 5), (0, 2, 4, 4), (1, 2, 1, 3), (1, 3, 6, 4), (2, 3, 3, 4),
        (2, 4, 5, 3), (3, 4, 1, 3), (3, 5, 4, 4), (4, 5, 2, 4), (1, 4, 7, 2)]
n = 6
supply = np.array([6, 0, 0, 0, -3, -3], float)
m = len(ARCS)
c = np.array([a[2] for a in ARCS], float)
u = np.array([a[3] for a in ARCS], float)

Ninc = np.zeros((n, m))
for k, (i, j, _, _) in enumerate(ARCS):
    Ninc[i, k] = 1; Ninc[j, k] = -1
assert supply.sum() == 0

In [ ]:
res = linprog(c, A_eq=Ninc, b_eq=supply, bounds=list(zip(np.zeros(m), u)), method="highs")
x = res.x
print("min cost (LP):", res.fun)
for k, (i, j, cij, uij) in enumerate(ARCS):
    if x[k] > 1e-9:
        print(f"  {i}->{j}: flow {x[k]:.0f} / {uij}   cost {cij}")

G = nx.DiGraph()
for v in range(n):
    G.add_node(v, demand=-supply[v])          # networkx: demand = inflow - outflow
for i, j, cij, uij in ARCS:
    G.add_edge(i, j, weight=cij, capacity=uij)
nx_cost = nx.min_cost_flow_cost(G)
print("min cost (networkx):", nx_cost)
assert np.isclose(res.fun, nx_cost)

## Check the optimality certificate

HiGHS reports equality-constraint marginals $\pi=\partial(\text{objective})/\partial b$. Then $\bar c=c-N^\top\pi$, which for arc $(i,j)$ is $c_{ij}-\pi_i+\pi_j$.

In [ ]:
pi = res.eqlin.marginals
rc = c - Ninc.T @ pi
print("potentials:", pi.round(3))
tol = 1e-7
for k, (i, j, _, _) in enumerate(ARCS):
    at_lower = x[k] < tol
    at_upper = x[k] > u[k] - tol
    if not at_lower and not at_upper:
        assert abs(rc[k]) < 1e-6, (k, rc[k])
    if at_lower and not at_upper:
        assert rc[k] > -1e-6
    if at_upper and not at_lower:
        assert rc[k] < 1e-6
    print(f"  {i}->{j}: x={x[k]:.0f}  reduced cost={rc[k]:+.3f}")
print("reduced-cost optimality conditions verified")

## Interpreting $\pi$

$\pi_v$ is the marginal cost of one more unit of supply at $v$ (with matching demand elsewhere absorbed by the system). Change `supply` by one unit at node 0 and node 5, re-solve, and compare the cost change with $\pi_0-\pi_5$.

In [ ]:
b2 = supply.copy(); b2[0] += 1; b2[5] -= 1
res2 = linprog(c, A_eq=Ninc, b_eq=b2, bounds=list(zip(np.zeros(m), u)), method="highs")
print("actual cost change:", res2.fun - res.fun, "   predicted pi_0 - pi_5:", pi[0] - pi[5])
assert np.isclose(res2.fun - res.fun, pi[0] - pi[5])

## Exercises

1. Formulate the **assignment problem** (4 workers, 4 jobs) as min-cost flow and check it against `scipy.optimize.linear_sum_assignment`.
2. Implement **successive shortest paths** (repeatedly route along a shortest path in the residual graph, with reduced costs to keep arc lengths non-negative).
3. Shrink a capacity until the predicted marginal from $\pi$ stops being valid. Explain this with degeneracy and basis change.